In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develope a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-05-29
Last Modified: 2026-05-29
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
"""WEDNESDAY"""
# unit and sanity checks!!!
# ---------*
# # cases
# 1. regular, no filtering
# 2. only mb or mf trials (balance and no balance)
# 3. both mb and mf trials, balanced
# 4. use passed idx_subsamps
# ---------*

# change the tent fitting to use the all trials fi - DONEt baseline
# plot the beta weight distributions across sessions
# rerun strategy balancing analyses, epochs - DONE
# use the same axes filter and look at those psths
# add movement (average normalized me on a trial)
"""--------------------------------------------"""
# add time
# one regressor

## both

In [ ]:
"""TODO: inform michael"""
# no outlier trial filtering at the moment
# for session subsampling, i divided the number of tents so that the frequency of slow drift is the same

In [ ]:
from sg.models import Encoder

encoder = Encoder(subj_id, sess_id, num_tents=12)
encoder.fit_encoder()
encoder.encoder_predict()

In [ ]:
encoder.verify(subtract_baseline=False)

In [ ]:
from squiggs.renderers import PETHWeightRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR
from core.data import get_psths_cond, get_choice_ts, get_tavg_sc_cond

"""
drift: 21, 29, 35, 36
response: 16, 25, 26, 28
"""

reg = "DLS"
mode = "response"

sc_tavg = get_tavg_sc_cond(
    encoder.robs[:, encoder.reg_idxs[reg]], encoder.trial_data, cond=mode
)

r = PETHWeightRenderer(
    weights=encoder.encoder.coef_[encoder.reg_idxs[reg], :],
    weight_names=encoder.dm_names,
    robs=encoder.robs[:, encoder.reg_idxs[reg]],
    sc_tavg=sc_tavg,
    event_times=get_choice_ts(encoder.trial_data, mode=mode),
    spike_times=encoder.spike_times[reg],
    peths=get_psths_cond(encoder.psths[reg], encoder.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
)

nv = NeuronViewer(num_units=len(encoder.psths[reg]), render_func=r, fig_dir=FIGURES_DIR)

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "strategy",
        "response_prev",
        "rewarded_prev",
    ],
)
se.plot_cvr2()
se.plot_dr2()

# strategy split

## aggregate

In [ ]:
# TODO
# 0. clean up plotting fn
# 1. aggregate across sessions

In [ ]:
import numpy as np
from core.data import subject_ids, session_ids

sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]
coefs = {"mb": {"DLS": [], "DMS": []}, "mf": {"DLS": [], "DMS": []}}
regressors = np.array(
    ["response_left", "response_right", "rewarded_incorr", "rewarded_corr"]
)

for sess_id in sess_ids:
    encoder_mb = Encoder(subj_id, sess_id, strategy_filter="mb", num_tents=12)
    encoder_mf = Encoder(subj_id, sess_id, strategy_filter="mf", num_tents=12)

    try:
        encoder_mb.fit_encoder()
        encoder_mf.fit_encoder()
    except RuntimeError:
        continue

    # get coefs and separate by region as well
    coefs_mb = encoder_mb.encoder.coef_
    coefs_mf = encoder_mf.encoder.coef_

    tv_idxs = [
        i for i, dm_name in enumerate(encoder_mb.dm_names) if dm_name in regressors
    ]

    coefs_mb_ = coefs_mb[:, tv_idxs]
    coefs_mf_ = coefs_mf[:, tv_idxs]

    coefs["mb"]["DLS"].extend(coefs_mb_[encoder_mb.reg_idxs["DLS"]])
    coefs["mb"]["DMS"].extend(coefs_mb_[encoder_mb.reg_idxs["DMS"]])
    coefs["mf"]["DLS"].extend(coefs_mf_[encoder_mb.reg_idxs["DLS"]])
    coefs["mf"]["DMS"].extend(coefs_mf_[encoder_mb.reg_idxs["DMS"]])

coefs = {
    strategy: {region: np.array(coefs[strategy][region]) for region in coefs[strategy]}
    for strategy in coefs
}

In [ ]:
coefs["mb"]["DMS"].shape

In [ ]:
region = "DLS"
regr = "response"
val = "right"


def plot_bweight_strategy(reg, regr, val):
    regr_idx = np.where(regressors == f"{regr}_{val}")[0]

    plt.figure(figsize=(2.5, 2), tight_layout=True)
    plt.scatter(
        coefs["mb"][reg][:, regr_idx], coefs["mf"][reg][:, regr_idx], s=0.5, alpha=0.5
    )

    plt.plot([-1.5, 1.5], [-1.5, 1.5], linewidth=0.5, linestyle="--", color="#666666")
    plt.axhline(y=0, color="k")
    plt.axvline(x=0, color="k")

    plt.xlabel(rf"mb $\beta$ {regr}_{val}")
    plt.ylabel(rf"mf $\beta$ {regr}_{val}")

    plt.show()


plot_bweight_strategy(reg="DLS", regr="response", val="left")

In [ ]:
plot_bweight_strategy(reg="DMS", regr="response", val="left")

In [ ]:
plot_bweight_strategy(reg="DLS", regr="response", val="right")
plot_bweight_strategy(reg="DMS", regr="response", val="right")

In [ ]:
plot_bweight_strategy(reg="DLS", regr="rewarded", val="corr")
plot_bweight_strategy(reg="DMS", regr="rewarded", val="corr")

In [ ]:
plot_bweight_strategy(reg="DLS", regr="rewarded", val="incorr")
plot_bweight_strategy(reg="DMS", regr="rewarded", val="incorr")

## single session

In [ ]:
encoder = Encoder(subj_id, sess_id, num_tents=5)
encoder.fit_encoder()
encoder.encoder_predict()
encoder.verify(subtract_baseline=True)

In [ ]:
encoder_mb = Encoder(subj_id, sess_id, strategy_filter="mb", num_tents=12)
encoder_mb.fit_encoder()

encoder_mf = Encoder(subj_id, sess_id, strategy_filter="mf", num_tents=12)
encoder_mf.fit_encoder()

In [ ]:
encoder_mf.verify()

In [ ]:
# weight comparison
import numpy as np

regr = "response"
val = "left"

i = np.where(encoder.dm_names == f"{regr}_{val}")[0][0]

plt.figure(tight_layout=True)
plt.scatter(
    encoder_mb.encoder.coef_[:, i], encoder_mf.encoder.coef_[:, i], s=0.5, alpha=0.5
)

plt.axhline(y=0, color="k")
plt.axvline(x=0, color="k")
plt.xlabel(f"mb, bweight {regr} {val}")
plt.ylabel(f"mf, bweight {regr} {val}")
plt.plot()